# Media Lab: transcribe a whole folder

Set a folder path below and run the cells from top to bottom. Every **video and audio file** inside it, including all sub-folders, gets:

- `transcript.txt` (clean paragraphs), `transcript_ts.txt` (a `[hh:mm:ss]` line every few seconds), `segments.json`, `meta.json`
- **videos:** `frames/` plus contact sheets `sheet_01.jpg`, ... (20 frames each), and `ocr.txt` with the slide and on-screen text
- **videos longer than 3 minutes:** `outline.md`, a timeline of each visual section with what was on screen and what was said
- one `_all_transcripts.md` per folder with every transcript in one file

**How frames are chosen**
- Up to 3 minutes: one frame about every 10 seconds.
- Longer videos: a quick scan finds where the picture actually changes (new slide, screen share, camera switch), and one frame is kept per change once the slide is fully built.
- Repeats (the speaker view, or a slide shown again) are dropped. A live webcam tile in the corner doesn't count as a change.
- Result: a 2-hour talking-head call gives a handful of frames, and a slide webinar gives about one frame per slide (at most 150).

**Speed:** about 15x real time on the 4090, so 100 hours of video takes roughly 7 hours.

**Where results go:** `media_lab/out/<folder name>/...`, mirroring your folder structure.

**Safety**
- Your files are only read. Nothing is moved, renamed, deleted or written next to them.
- It uses the **RTX 4090 only**, capped at 25% VRAM, and runs one GPU job at a time at low CPU priority.
- When it finishes (or you stop it), it frees the GPU and its temp files. You can close this notebook while it runs.
- Re-running the same folder skips files that are already done, so it's safe to stop and resume.

In [27]:
FOLDER = "/home/pavlo/lab/TonyRobins"   # <- the only thing to change

import sys; sys.path.insert(0, "/home/pavlo/lab/media_lab")
import importlib, media_lab as ml; importlib.reload(ml)
ml.preview(FOLDER)          # lists files, total hours and a time estimate (nothing runs yet)

/home/pavlo/lab/TonyRobins
  media: 1083 files, 31.82 GB, 153.1 h, 0 already done
  learning materials: 22 files (copied + text extracted)
  still uploading right now: 0
  rough time: ~7.7-12.8 h (whisper runs ~12-20x faster than real time)
  Creating_Lasting_Change/Day 1 – The Art of Leadership/1. A Journey into the Human Psyche 920 Breakthrough Creating Las.mp3
  Creating_Lasting_Change/Day 1 – The Art of Leadership/2. Why People Do What They Do 1627 Breakthrough Creating Lasting.mp3
  Creating_Lasting_Change/Day 1 – The Art of Leadership/3. The 3 Forces that Shape Our Destiny 1714 Breakthrough Creatin.mp3
  Creating_Lasting_Change/Day 1 – The Art of Leadership/4. The Formula for Happiness 1237 Breakthrough Creating Lasting.mp3
  Creating_Lasting_Change/Day 1 – The Art of Leadership/5. Understand the Invisible Forces that Shape Our Lives 1744 Bre.mp3
  Creating_Lasting_Change/Day 10 – The Depth & Breadth of Leadership/1. Your Challenge as a Leader 1623 Breakthrough Creating Lasting.m

In [28]:
print(ml.run_folder(FOLDER))  # starts the background job and returns immediately

started pid 1113737: 1083 media + 22 learning materials in /home/pavlo/lab/TonyRobins
check progress with ml.status()


In [35]:
ml.status()         # re-run any time: progress, speed, ETA and the last log lines

○ FINISHED
folder   : /home/pavlo/lab/TonyRobins
started  : 2026-10-21 11:26:39   (running for -)   finished: 2026-10-21 20:20:04
media    : 1083 done this run · 0 already done earlier · 0 failed · 0 uploading · of 1083
materials: 22 copied this run · 0 already copied · 0 failed · of 22
hours    : 153.09 h transcribed this run · 0.0 h left · speed 17.2x · ETA 0.0 h

--- log of this run (last 15 of 2192 lines; ml.log() for all) ---
   ok 14757w, asr 22.6x, frames 143 (keyframes), 260.4s
20:14:41 media Ultimate_Relationship/8. Day 7 - The Power of Connection/1. Rekindling Intimacy with Your Lover Arantza amp Alan 10709 Br.mp4 01:07:08
   ok 9799w, asr 23.3x, frames 100 (keyframes), 176.2s
20:17:38 media Ultimate_Relationship/8. Day 7 - The Power of Connection/2. Heart Meditation 248 Breakthrough Ultimate Relationship.mp3 00:02:49
   ok 350w, asr 26.6x, frames - (audio), 6.5s
20:17:44 media Ultimate_Relationship/9. Day 8 - Understand & Appreciate the One You Love/1. The Vital Importance o

In [19]:
ml.overview(FOLDER)          # table of finished files (duration, words, wpm, frames)

47 media files · 45 learning materials (45 with text) · see /home/pavlo/lab/media_lab/out/Yap Challenge/_manifest.md, _all_transcripts.md, _all_materials.md


,rel,kind,duration_hms,words,wpm,frames,visual_mode,asr_speed,seconds
0,01 - Day 1 - LIVE Kickoff Call + Replay/01 - D...,video,02:25:44,25973,178.2,101,keyframes,20.3,443.0
1,02 - Day 2 - Your Content Game Plan Training +...,video,00:54:58,11263,204.9,64,keyframes,18.3,197.7
2,03 - Day 3 - The Art of a Damn Good Yap Master...,video,00:41:03,7932,193.2,57,keyframes,19.3,134.8
3,04 - Bonus - Optimizing (tools tech set-up mon...,video,00:32:08,6881,214.0,19,keyframes,17.6,112.6
4,05 - Bonus - Lets Plan Your Content Calendar/0...,video,00:27:19,5645,206.6,36,keyframes,18.1,101.0
5,06 - Bonus - Pick Your Content Formats/06 - Bo...,video,00:24:22,4853,199.1,7,keyframes,18.7,80.5
6,07 - Day 4 - Daily Drop + Dare BEGINS! letssss...,video,00:14:51,2871,193.2,5,keyframes,19.2,47.2
7,08 - Day 5 - THE DARE post your unique contrar...,video,00:14:14,2858,200.6,5,keyframes,18.5,47.2
8,09 - Day 6 - THE DARE post for your audience's...,video,00:11:40,2390,204.7,4,keyframes,18.0,39.8
9,10 - Day 7 - THE DARE create an identity-based...,video,00:16:55,3299,195.0,4,keyframes,18.7,55.3


In [8]:
# print(ml.read(FOLDER, "module 1/call.mp4"))                  # one transcript
# print(ml.read(FOLDER, "module 1/call.mp4", "outline.md"))    # slides + what was said, per section
# ml.stop()              # stop the job (it cleans up after itself)
ml.stop()    

'sent stop to pid 211887'

In [ ]:
FOLDER_2 = "/home/pavlo/lab/Kate Buck Jr - Traffic You Earn" 
print(ml.run_folder(FOLDER_2))